# Incremental ontology updates — diff a Protégé edit, patch with SPARQL

The workflow: an ontology is loaded in Fuseki, you edit the TTL in **Protégé**, and want
only the *changes* pushed back — as SPARQL `DELETE`/`INSERT`, not a full reload. This is
`ontology_modeler`'s `OntologyDiffer` (pure diff) and `GraphSynchronizer` (apply + verify +
fallback), reached via `OntologyModeler().sync`.

### The one hard problem: blank nodes

An RDF graph is a **set of triples**, so an update is a set difference — in principle. Two
things break the naïve version:

1. **Never diff the TTL as text.** Protégé reorders subjects and rewrites prefixes;
   none of that changes a triple. Diff the *parsed graphs*.
2. **Blank nodes.** OWL writes every `owl:Restriction` and list as an anonymous subgraph,
   and Protégé reassigns their ids on every save, so a naïve triple diff reports unchanged
   restrictions as deleted-and-re-added. `OntologyDiffer` is **blank-node aware**: ground
   triples diff by set difference; anonymous structures diff per named anchor by
   isomorphism. Deletions of anonymous structures use `DELETE … WHERE` (blank nodes →
   variables), because SPARQL forbids blank nodes in `DELETE DATA`.

Demo writes go into `urn:graph:__hbim_demo__/` and are dropped; FIBO is only read. Run from
`src/Ontology Modeler/notebooks/`.

In [1]:
import sys
from pathlib import Path
from rdflib import Graph
MODULE_DIR = Path.cwd().parent
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from ontology_modeler import OntologyModeler, OntologyDiffer, UpdatePlan
from ontology_modeler.diff import has_bnode

om = OntologyModeler()
client = om.fuseki
sync = om.sync                         # a GraphSynchronizer over the shared client
DEMO = "urn:graph:__hbim_demo__/hbim.ttl"
print("Fuseki up at", client.settings.base_url, "->", client.ping())

Fuseki up at http://localhost:3030 -> 2026-07-17T06:59:17.537+00:00


## 1. The starting point — an HBIM ontology in the store

A small but realistic HBIM ontology, including an **anonymous restriction** on `Wall` (the
blank node that trips up naïve diffing).

In [2]:
HBIM_V1 = """@prefix :    <http://hbim.example/ontology/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:HBIM a owl:Ontology ; owl:versionInfo "1.0.0" .

:BuildingElement a owl:Class ; rdfs:label "building element" .
:Wall  a owl:Class ; rdfs:label "wall"  ; rdfs:subClassOf :BuildingElement ,
        [ a owl:Restriction ; owl:onProperty :hasMaterial ; owl:someValuesFrom :Material ] .
:LoadBearingWall a owl:Class ; rdfs:label "load-bearing wall" ; rdfs:subClassOf :Wall .
:Floor a owl:Class ; rdfs:label "floor" ; rdfs:subClassOf :BuildingElement .
:Roof  a owl:Class ; rdfs:label "roof"  ; rdfs:subClassOf :BuildingElement .
:Door  a owl:Class ; rdfs:label "door"  ; rdfs:subClassOf :BuildingElement .
:Window a owl:Class ; rdfs:label "window" ; rdfs:subClassOf :BuildingElement .
:Beam  a owl:Class ; rdfs:label "beam"  ; rdfs:subClassOf :BuildingElement .
:Space a owl:Class ; rdfs:label "space" .

:Material a owl:Class ; rdfs:label "material" .
:Brick    a owl:Class ; rdfs:label "brick"    ; rdfs:subClassOf :Material .
:Concrete a owl:Class ; rdfs:label "concrete" ; rdfs:subClassOf :Material .
:Steel    a owl:Class ; rdfs:label "steel"    ; rdfs:subClassOf :Material .
:Timber   a owl:Class ; rdfs:label "timber"   ; rdfs:subClassOf :Material .

:hasMaterial  a owl:ObjectProperty   ; rdfs:label "has material" ; rdfs:domain :BuildingElement ; rdfs:range :Material .
:isPartOf     a owl:ObjectProperty   ; rdfs:label "is part of"   ; rdfs:domain :BuildingElement ; rdfs:range :BuildingElement .
:hasLocation  a owl:ObjectProperty   ; rdfs:label "has location" ; rdfs:domain :BuildingElement ; rdfs:range :Space .
:adjacentTo   a owl:ObjectProperty   ; rdfs:label "adjacent to"  ; rdfs:domain :BuildingElement ; rdfs:range :BuildingElement .
:hasThickness a owl:DatatypeProperty ; rdfs:label "has thickness"; rdfs:domain :Wall ; rdfs:range xsd:decimal .
:hasWidth     a owl:DatatypeProperty ; rdfs:label "has width"    ; rdfs:domain :BuildingElement ; rdfs:range xsd:decimal .
:deprecatedProp a owl:DatatypeProperty ; rdfs:label "legacy — to be removed" .
"""

g_v1 = Graph().parse(data=HBIM_V1, format="turtle")
client.put_graph(DEMO, g_v1)
print(f"loaded HBIM v1: {len(client.get_graph(DEMO))} triples")

loaded HBIM v1: 71 triples


## 2. The Protégé edit (simulated)

A fresh serialization — its blank-node ids differ even for untouched parts. Genuine changes:
version bump; `LoadBearingWall` label edit; delete `deprecatedProp`; add `hasHeight`; add
`Column` **with a restriction**; change `Wall`'s restriction `someValuesFrom → allValuesFrom`.
The last two exercise the blank-node handling.

In [3]:
HBIM_V2 = """@prefix :    <http://hbim.example/ontology/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

:HBIM a owl:Ontology ; owl:versionInfo "1.1.0" .

:BuildingElement a owl:Class ; rdfs:label "building element" .
:Wall  a owl:Class ; rdfs:label "wall"  ; rdfs:subClassOf :BuildingElement ,
        [ a owl:Restriction ; owl:onProperty :hasMaterial ; owl:allValuesFrom :Material ] .
:LoadBearingWall a owl:Class ; rdfs:label "load-bearing wall (structural)" ; rdfs:subClassOf :Wall .
:Floor a owl:Class ; rdfs:label "floor" ; rdfs:subClassOf :BuildingElement .
:Roof  a owl:Class ; rdfs:label "roof"  ; rdfs:subClassOf :BuildingElement .
:Door  a owl:Class ; rdfs:label "door"  ; rdfs:subClassOf :BuildingElement .
:Window a owl:Class ; rdfs:label "window" ; rdfs:subClassOf :BuildingElement .
:Beam  a owl:Class ; rdfs:label "beam"  ; rdfs:subClassOf :BuildingElement .
:Space a owl:Class ; rdfs:label "space" .
:Column a owl:Class ; rdfs:label "column" ; rdfs:subClassOf :BuildingElement ,
        [ a owl:Restriction ; owl:onProperty :hasMaterial ; owl:someValuesFrom :Material ] .

:Material a owl:Class ; rdfs:label "material" .
:Brick    a owl:Class ; rdfs:label "brick"    ; rdfs:subClassOf :Material .
:Concrete a owl:Class ; rdfs:label "concrete" ; rdfs:subClassOf :Material .
:Steel    a owl:Class ; rdfs:label "steel"    ; rdfs:subClassOf :Material .
:Timber   a owl:Class ; rdfs:label "timber"   ; rdfs:subClassOf :Material .

:hasMaterial  a owl:ObjectProperty   ; rdfs:label "has material" ; rdfs:domain :BuildingElement ; rdfs:range :Material .
:isPartOf     a owl:ObjectProperty   ; rdfs:label "is part of"   ; rdfs:domain :BuildingElement ; rdfs:range :BuildingElement .
:hasLocation  a owl:ObjectProperty   ; rdfs:label "has location" ; rdfs:domain :BuildingElement ; rdfs:range :Space .
:adjacentTo   a owl:ObjectProperty   ; rdfs:label "adjacent to"  ; rdfs:domain :BuildingElement ; rdfs:range :BuildingElement .
:hasThickness a owl:DatatypeProperty ; rdfs:label "has thickness"; rdfs:domain :Wall ; rdfs:range xsd:decimal .
:hasWidth     a owl:DatatypeProperty ; rdfs:label "has width"    ; rdfs:domain :BuildingElement ; rdfs:range xsd:decimal .
:hasHeight    a owl:DatatypeProperty ; rdfs:label "has height"   ; rdfs:domain :BuildingElement ; rdfs:range xsd:decimal .
"""

g_v2 = Graph().parse(data=HBIM_V2, format="turtle")
print(f"edited HBIM v2: {len(g_v2)} triples (in memory, not yet applied)")

edited HBIM v2: 80 triples (in memory, not yet applied)


## 3. Why the naïve diff is wrong

Set-difference over *all* triples over-counts: the `Wall` restriction only *changed*, yet
blank-node relabelling makes whole anonymous structures appear on both sides.

In [4]:
naive_del, naive_add = OntologyDiffer.naive_diff(g_v1, g_v2)
print(f"naïve set-difference: {len(naive_del)} deletions, {len(naive_add)} additions\n")
print("blank-node 'deletions' that are really just relabelled:")
for t in sorted(naive_del, key=str):
    if has_bnode(t):
        print("  -", t[1].split('#')[-1].split('/')[-1])

naïve set-difference: 8 deletions, 17 additions

blank-node 'deletions' that are really just relabelled:
  - type
  - onProperty
  - someValuesFrom
  - subClassOf


## 4. Blank-node-aware diff → an `UpdatePlan`

`OntologyDiffer.plan` diffs ground triples by set difference and handles anonymous
structures per named anchor: for each subject whose blank-node closure changed, delete the
old closure and insert the new one.

In [5]:
plan = OntologyDiffer.plan(g_v1, g_v2, DEMO)
print(plan.summary())
print("\nground deletions:")
for t in sorted(plan.ground_del, key=str):
    print("  -", t[0].split('/')[-1], t[1].split('#')[-1].split('/')[-1], repr(str(t[2]))[:40])
print("\nanonymous structures replaced (anchor):", [a.split('/')[-1] for a, _ in plan.del_stars])
print("anonymous structures inserted (anchor):", [a.split('/')[-1] for a, _ in plan.add_stars])

ground deletions : 4
ground additions : 9
anonymous structures deleted (anchors): 1
anonymous structures inserted (anchors): 2
total triples changed: 25

ground deletions:
  - HBIM versionInfo '1.0.0'
  - LoadBearingWall label 'load-bearing wall'
  - deprecatedProp type 'http://www.w3.org/2002/07/owl#DatatypeP
  - deprecatedProp label 'legacy — to be removed'

anonymous structures replaced (anchor): ['Wall']
anonymous structures inserted (anchor): ['Column', 'Wall']


## 5. The generated SPARQL

One request, `;`-separated so Fuseki runs it as **one transaction**: `DELETE DATA` (ground),
`WITH … DELETE … WHERE` (the old `Wall` restriction, blank node as `?b0`, anchored on
`<Wall>`), and `INSERT DATA` (additions, with fresh blank nodes for new restrictions).

In [6]:
print(plan.to_sparql())

DELETE DATA {
  GRAPH <urn:graph:__hbim_demo__/hbim.ttl> {
    <http://hbim.example/ontology/HBIM> <http://www.w3.org/2002/07/owl#versionInfo> "1.0.0" .
    <http://hbim.example/ontology/LoadBearingWall> <http://www.w3.org/2000/01/rdf-schema#label> "load-bearing wall" .
    <http://hbim.example/ontology/deprecatedProp> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#DatatypeProperty> .
    <http://hbim.example/ontology/deprecatedProp> <http://www.w3.org/2000/01/rdf-schema#label> "legacy — to be removed" .
  }
} ;
WITH <urn:graph:__hbim_demo__/hbim.ttl>
DELETE {
    <http://hbim.example/ontology/Wall> <http://www.w3.org/2000/01/rdf-schema#subClassOf> ?b0 .
    ?b0 <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://www.w3.org/2002/07/owl#Restriction> .
    ?b0 <http://www.w3.org/2002/07/owl#onProperty> <http://hbim.example/ontology/hasMaterial> .
    ?b0 <http://www.w3.org/2002/07/owl#someValuesFrom> <http://hbim.example/ontology/Material> .
}
WHER

## 6. Dry run — decide before touching the store

`sync.sync(dry_run=True)` builds the plan and picks a strategy without applying. Rule: if the
change ratio exceeds `fallback_threshold` (default 0.5) or blank nodes are unsafe, replace the
whole graph with `PUT`. This edit is small → **incremental**.

In [7]:
preview = sync.sync(DEMO, g_v2, dry_run=True)
print(f"chosen method : {preview.method}")
print(f"change ratio  : {preview.ratio:.2f}  (threshold 0.50)")
print(f"triples changed: {preview.plan.changed_triples} of {len(g_v1)} in the old graph")

chosen method : incremental
change ratio  : 0.35  (threshold 0.50)
triples changed: 25 of 71 in the old graph


## 7. Apply atomically, then verify

`sync.sync` sends the one-transaction update, then **fetches the graph back and checks it is
isomorphic to the target** — the definitive proof, blank nodes and all. A failed incremental
patch falls back to `PUT`, so the store always converges.

In [8]:
result = sync.sync(DEMO, g_v2)          # old side fetched from Fuseki
print(f"method   : {result.method}")
print(f"applied  : {result.applied}")
print(f"verified : {result.verified}  (graph is isomorphic to HBIM v2)")

method   : incremental
applied  : True
verified : True  (graph is isomorphic to HBIM v2)


In [9]:
def ask(q):
    return client.select("PREFIX : <http://hbim.example/ontology/> "
                         "PREFIX owl: <http://www.w3.org/2002/07/owl#> "
                         "PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> " + q)
print("versionInfo now:", ask(f"SELECT ?v WHERE {{ GRAPH <{DEMO}> {{ :HBIM owl:versionInfo ?v }} }}")[0]["v"])
print("Column added (label):", ask(f"SELECT ?l WHERE {{ GRAPH <{DEMO}> {{ :Column rdfs:label ?l }} }}")[0]["l"])
print("deprecatedProp removed:", len(ask(f"SELECT ?p WHERE {{ GRAPH <{DEMO}> {{ :deprecatedProp ?p ?o }} }}")) == 0)
print("Wall restriction now allValuesFrom:",
      len(ask(f"SELECT ?r WHERE {{ GRAPH <{DEMO}> {{ :Wall rdfs:subClassOf ?r . ?r owl:allValuesFrom :Material }} }}")) == 1)

versionInfo now: 1.1.0
Column added (label): column
deprecatedProp removed: True


Wall restriction now allValuesFrom: True


## 8. Idempotency

`DELETE DATA` of an absent triple and `INSERT DATA` of a present one are no-ops, so re-syncing
the same target produces an **empty plan**.

In [10]:
again = sync.sync(DEMO, g_v2)
print(f"second sync — triples changed: {again.plan.changed_triples}, verified: {again.verified}")

second sync — triples changed: 0, verified: True


## 9. The fallback — guaranteed convergence

Incremental patching is the optimization; correctness is guaranteed by a full-graph `PUT`
fallback, taken on large diffs or unsafe blank nodes. Force it here with a tiny threshold.

In [11]:
client.put_graph(DEMO, g_v1)                        # reset to v1
forced = sync.sync(DEMO, g_v2, fallback_threshold=0.01)
print(f"method   : {forced.method}   (ratio {forced.ratio:.2f} > 0.01 threshold)")
print(f"verified : {forced.verified}")

method   : put   (ratio 0.35 > 0.01 threshold)
verified : True


## 10. "Old" side: Fuseki vs a saved TTL

By default `sync` fetches the old side **from Fuseki** — the store as it actually is. You can
instead pass an explicit `old_graph` from your previous TTL snapshot: faster and offline, but
a stale snapshot risks a patch built against triples that aren't there (which the isomorphism
check then catches).

In [12]:
from_fuseki = sync.sync(DEMO, g_v2, dry_run=True)
from_file   = sync.sync(DEMO, g_v2, old_graph=g_v1, dry_run=True)
print("old=Fuseki  -> changed:", from_fuseki.plan.changed_triples)
print("old=TTL file-> changed:", from_file.plan.changed_triples)

old=Fuseki  -> changed: 0
old=TTL file-> changed: 25


## Summary — and the agent tool

1. **Old side** = the store (`client.get_graph`) or a saved TTL.
2. **Blank-node-aware diff** (`OntologyDiffer.plan`) → ground by set difference; anonymous
   structures per named anchor by isomorphism.
3. **One atomic SPARQL update** — `DELETE DATA` + `DELETE … WHERE` + `INSERT DATA`.
4. **Verify isomorphic**; fall back to `PUT` on large diffs, unsafe blank nodes, or a failed
   verification. Idempotent by construction.

```python
from ontology_modeler import OntologyModeler
from rdflib import Graph
om = OntologyModeler()

def tool_sync_ontology(graph_uri: str, new_ttl: str) -> dict:
    "Apply a new ontology version to Fuseki incrementally; returns method + verified."
    r = om.sync.sync(graph_uri, Graph().parse(data=new_ttl, format="turtle"))
    return {"method": r.method, "verified": r.verified, "changed": r.plan.changed_triples}
```

In [13]:
# cleanup — drop the demo graph, confirm FIBO is untouched
client.delete_graph(DEMO)
rows = client.select("SELECT (COUNT(DISTINCT ?g) AS ?graphs) (COUNT(*) AS ?t) WHERE { GRAPH ?g { ?s ?p ?o } }")[0]
print(f"demo graph dropped. store now: {rows['graphs']} graphs, {rows['t']} triples (FIBO intact)")

demo graph dropped. store now: 299 graphs, 133166 triples (FIBO intact)
